# Info-Driven Bar Trading — Run on Colab

> ⚠️ **This repository and notebook were created entirely by AI.**

Reproduces information-driven bars (CUSUM / range) + triple-barrier labeling + a ResNet-LSTM, under a quarterly expanding-window protocol.

**Before running:** `Runtime → Change runtime type → T4 GPU`. This notebook assumes your gold `datasets/` folder (the per-config feature parquets) is in **Google Drive**.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/fedallah-jr/Info-Driven-Bar-Trading.git
%cd Info-Driven-Bar-Trading

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Put the gold datasets where the code expects them (`data/datasets/`)

The pipeline reads `data/datasets/<coin>_<bartype>_thr<θ>.parquet` relative to the repo root. **Edit `DRIVE_DATASETS`** to your folder, then copy the parquets in.

In [ ]:
DRIVE_DATASETS = '/content/drive/MyDrive/datasets'   # <-- set to your datasets/ folder in Drive
!mkdir -p data/datasets
!cp -v {DRIVE_DATASETS}/*.parquet data/datasets/
print('available datasets:')
!ls data/datasets/ | wc -l
!ls data/datasets/ | head

## 4. Confirm GPU + dependencies
(Colab ships torch / pandas / numpy / sklearn / pyarrow — no installs needed.)

In [ ]:
import torch, pandas, numpy, sklearn, pyarrow
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 5. Quick smoke test (~1–2 min)
Confirms the whole chain runs before the long job — one config, one quarter, a few epochs.

In [ ]:
!python3 model/train.py --coin ETHUSDT --bartype cusum --threshold 0.02 --barrier 0.05 \
        --quarters 2022Q4 --seeds 0 --epochs 5

## 6. Full v1 grid (the real run)

ETH/BTC × {CUSUM, range} × 2%, 5 quarters × 3 seeds. Roughly tens of minutes to ~2 h on a T4 (early stopping shortens it). For just the headline cell, use the commented single-config line instead.

In [ ]:
!python3 model/train.py --grid
# Or a single config:
# !python3 model/train.py --coin ETHUSDT --bartype cusum --threshold 0.02 --barrier 0.05

## 7. Evaluate → metrics table

Look for the `ETHUSDT_cusum_thr0.020…` row — the paper's target is **≈ +91.6% Net P/L, 1.42 Sharpe** over 2022-Q2 … 2023-Q2.

In [ ]:
!python3 model/evaluate.py
from IPython.display import Markdown
Markdown(open('results/model/RESULTS.md').read())

## 8. (Optional) Backtest detail + save results to Drive
Colab sessions are ephemeral — copy results to Drive so they persist.

In [ ]:
!python3 model/backtest.py results/model/ETHUSDT_cusum_thr0.020_triple_barrier_resnet_lstm_preds.parquet
!cp -r results /content/drive/MyDrive/info-driven-results

### Notes
- `--grid` trains only ETH/BTC at 2% (that's `v1_grid`). To train other coins you have datasets for: `!python3 model/train.py --coin SOLUSDT --bartype cusum --threshold 0.02 --barrier 0.025`. Per the paper, barrier/threshold don't transfer blindly across coins — treat non-ETH/BTC as exploratory.
- If step 3's `cp` finds nothing, your `DRIVE_DATASETS` path is wrong — run `!ls /content/drive/MyDrive` to locate the folder.
- To persist training outputs automatically, pass `--out-dir /content/drive/MyDrive/info-driven-results` to `train.py`.